# Information Retrieval in the 1914 French Press

## Fetching the Data

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh french-press-1914

In [ ]:
!ls french-press-1914/le_figaro/1914/

## Corpus

This corpus gathers the 1914 issues of two French daily newspapers, *L'Humanité* and *Le Figaro*, digitized by the Bibliothèque nationale de France. Each issue is a JSON file, at the path `<newspaper>/1914/<YYYYMMDD>/<YYYYMMDD>.metadata.fulltext.json`: one folder per newspaper, then one folder per publication date, which holds that issue's file.

The previous cell lists *Le Figaro*'s date folders because they fit on the screen: the corpus has only 24 issues of *Le Figaro*, between January 19 and November 18, against 360 of *L'Humanité*. This imbalance will show in the search results.

The text comes from raw optical character recognition (OCR), never corrected: `french-press-1914/README.md` warns that it has many errors, and the first issue of *Le Figaro* states a confidence of 0.57 (`format` field). Misread, split or merged words are therefore found in the paragraphs.

Three fields of the JSON files are used here:

- `contentAsText`, a list of strings, in which paragraphs are separated by line breaks (`\n`);
- `date`, the publication date, in a list of a single string (`content["date"][0]`);
- `title`, the newspaper's name, likewise (`content["title"][0]`).

The next cell loads the corpus into `data`, a 3-column `numpy.ndarray`, with one row per paragraph of more than 10 words:

- `data[:, 0]`, the paragraph's text, lowercased and stripped of punctuation by `preprocess_text`;
- `data[:, 1]`, the newspaper;
- `data[:, 2]`, the issue's date.

In [ ]:
import datetime
import json
import pathlib
import re

import numpy
import tqdm.notebook


def preprocess_text(text: str) -> str:
  return re.sub(r"\W+", " ", text).lower()


def get_data(directory: pathlib.Path) -> numpy.ndarray:
  rows = []
  for issue in tqdm.notebook.tqdm(list(directory.rglob("*.json"))):
    with open(issue, encoding="utf8") as file:
      content = json.load(file)
    newspaper = content["title"][0]
    date = datetime.date.fromisoformat(content["date"][0]).strftime("%d/%m/%Y")
    for block in content["contentAsText"]:
      for paragraph in block.split("\n"):
        paragraph = preprocess_text(paragraph)
        if len(paragraph.split()) > 10:
          rows.append((paragraph, newspaper, date))
  # dtype=object: a fixed-width string array would reserve for each cell the room of
  # the longest paragraph (8,434 characters), that is 4.9 GB for this corpus
  return numpy.array(rows, dtype=object)


data = get_data(pathlib.Path("french-press-1914"))
print(data.shape)

## TF-IDF Representation

Create a `TfidfVectorizer` in a variable `vectorizer`, then the matrix `X` of the corpus paragraphs (`data[:, 0]`) turned into TF-IDF weight vectors: one row per paragraph, one column per word of the vocabulary. The rest of the lab reuses `vectorizer` and `X`.

Then display the shape of `X` and its number of nonzero values: what share of the matrix's cells is nonzero? What does that tell you about how to store it?

**Useful documentation**

- [`sklearn.feature_extraction.text.TfidfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)
- [`scipy.sparse.csr_matrix`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.csr_matrix.html)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `sklearn.feature_extraction.text.TfidfVectorizer()`: the default parameters are enough. It lowercases the texts and only keeps the words of at least two characters
- `vectorizer.fit_transform(data[:, 0])`: learns the vocabulary and the idf on the paragraphs, then returns `X`, a sparse matrix (`scipy.sparse.csr_matrix`) that only stores the nonzero values
- `X.shape`: the number of rows and columns; `X.nnz`: the number of nonzero values
- to check: `X` has 48,333 rows, 486,005 columns and 5,642,466 nonzero values, that is 0.024% of the cells. Dense, in `float64`, it would take 188 GB; sparse, it takes 68 MB

A skeleton to copy into the exercise cell, replacing each `...`:

```python
import sklearn.feature_extraction.text

vectorizer = sklearn.feature_extraction.text.TfidfVectorizer()
X = ...
print("Shape of X:", ...)
print("Nonzero values:", ...)
print(f"Nonzero share: {...:.3%}")
```

### Solution

In [ ]:
import sklearn.feature_extraction.text

vectorizer = sklearn.feature_extraction.text.TfidfVectorizer()
X = vectorizer.fit_transform(data[:, 0])
print("Shape of X:", X.shape)
print("Nonzero values:", X.nnz)
print(f"Nonzero share: {X.nnz / (X.shape[0] * X.shape[1]):.3%}")

## A Noisy Corpus

Before searching the corpus, look at what it holds:

1. How many paragraphs does each newspaper have? What should you expect of a search's results?
2. What share of `vectorizer`'s vocabulary appears in a single paragraph only? These are the words with the highest idf (`vectorizer.idf_`).
3. Display 20 of these words, drawn at random: how many are correctly spelled words? Where do the others come from?

**Useful documentation**

- [`numpy.unique`](https://numpy.org/doc/stable/reference/generated/numpy.unique.html)
- [`sklearn.feature_extraction.text.TfidfVectorizer.get_feature_names_out`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html#sklearn.feature_extraction.text.TfidfVectorizer.get_feature_names_out)
- [`numpy.random.Generator.choice`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.choice.html)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `numpy.unique(data[:, 1], return_counts=True)`: the newspapers and, with `return_counts=True`, the number of paragraphs of each
- `vectorizer.vocabulary_`: the vocabulary, a dictionary mapping each word to its column in `X`; `len(vectorizer.vocabulary_)` gives its size
- `vectorizer.idf_`: each column's idf, which decreases with the number of paragraphs containing the word. `vectorizer.idf_ == vectorizer.idf_.max()` is therefore a boolean array, true for the words of a single paragraph, and its mean (`.mean()`) the share asked for
- `vectorizer.get_feature_names_out()`: the words, in the order of the columns; indexed by this boolean array, only the words of a single paragraph remain
- `numpy.random.default_rng(0).choice(words, 20, replace=False)`: 20 words drawn at random, without replacement; the seed `0` gives the same ones at every run
- to check: 41,998 paragraphs from *L'Humanité* and 6,335 from *Le Figaro*; 360,882 of the vocabulary's 486,005 words (74%) appear in a single paragraph only

A skeleton to copy into the exercise cell, replacing each `...`:

```python
newspapers, paragraph_counts = ...
print(newspapers, paragraph_counts)

single_paragraph = ...  # true for the words of a single paragraph
print(f"{single_paragraph.sum()} words out of {...}: {...:.0%}")

words = vectorizer.get_feature_names_out()
print(numpy.random.default_rng(0).choice(..., 20, replace=False))
```

### Solution

In [ ]:
newspapers, paragraph_counts = numpy.unique(data[:, 1], return_counts=True)
print(newspapers, paragraph_counts)

single_paragraph = vectorizer.idf_ == vectorizer.idf_.max()
print(
  f"{single_paragraph.sum()} words out of {len(vectorizer.vocabulary_)}: "
  f"{single_paragraph.mean():.0%}"
)

words = vectorizer.get_feature_names_out()
print(numpy.random.default_rng(0).choice(words[single_paragraph], 20, replace=False))

What these figures show:

- *L'Humanité* provides 87% of the paragraphs: most of a search's results will come from it, without *Le Figaro* covering the subject any less. It even has no issue between June 11 and August 7, 1914.
- 74% of the vocabulary appears in a single paragraph only. Among the 20 words drawn, most are words misread by the OCR (`tiavaux` for "travaux", `licbknedii` for "Liebknecht"), the others numbers or rare words (`maniabilité`).
- These words have the highest idf, the one the slide "TF-IDF — Why $df_i$?" gives to rare words, the most discriminative ones. Here, they discriminate nothing: no query contains them. And a paragraph in which the OCR misread a word of the query escapes the search by that word.

## Similarity

Implement a function `best_score(query: str) -> None` that:

- applies the paragraphs' preprocessing (`preprocess_text`) to the query, then turns it into a TF-IDF vector with `vectorizer`;
- computes the cosine similarity of each paragraph of `X` with the query;
- displays the 5 most similar paragraphs, from the first to the fifth: the newspaper, the date, the similarity, the paragraph's index in `data` and its text.

Test it with `best_score("attentat sarajevo")` (attack Sarajevo): the corpus is in French, so the queries stay in French. Which newspaper do the results come from, and why? Do you find the attack of June 28, 1914? What OCR error does the headline announcing the archduke's death contain?

**Useful documentation**

- [`sklearn.metrics.pairwise.cosine_similarity`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html)
- [`numpy.argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)

In [ ]:
def best_score(query: str) -> None:
  # Your code here
  ...


# Your code here

### Additional hints

The calls and parameters to use:

- `vectorizer.transform([preprocess_text(query)])`: the query's TF-IDF vector, a one-row matrix. `transform`, not `fit_transform`: the vocabulary and the idf stay those of the corpus. `TfidfVectorizer` would already lowercase the query and ignore its punctuation; `preprocess_text` keeps it processed like the paragraphs if that preprocessing changes (to remove accents, for instance)
- `sklearn.metrics.pairwise.cosine_similarity(X, query_vector)[:, 0]`: each paragraph's similarity with the query (a one-column matrix, which `[:, 0]` turns into an array)
- `(-similarities).argsort()[:5]`: `argsort` gives the indices that sort an array in increasing order; on the opposite of the similarities, the first 5 are those of the 5 largest
- `text, newspaper, date = data[index]`: the three columns of a row
- to check: on `attentat sarajevo`, the best similarity is 0.54

A skeleton to copy into the exercise cell, replacing each `...`:

```python
import sklearn.metrics.pairwise


def best_score(query: str) -> None:
  query_vector = ...
  similarities = ...
  for index in ...:  # the indices of the 5 largest similarities
    text, newspaper, date = data[index]
    print(
      f"{newspaper:10} on {date}; "
      f"similarity {similarities[index]:5.2f}; "
      f"index {index:6}; "
      f"{text}"
    )


best_score("attentat sarajevo")
```

### Solution

In [ ]:
import sklearn.metrics.pairwise


def best_score(query: str) -> None:
  # The same preprocessing as the paragraphs: TfidfVectorizer would already lowercase
  # the query, but query and paragraphs must stay processed the same way
  query_vector = vectorizer.transform([preprocess_text(query)])
  similarities = sklearn.metrics.pairwise.cosine_similarity(X, query_vector)[:, 0]
  # argsort sorts in increasing order: on the opposite of the similarities, the
  # indices of the largest come first
  for index in (-similarities).argsort()[:5]:
    text, newspaper, date = data[index]
    print(
      f"{newspaper:10} on {date}; "
      f"similarity {similarities[index]:5.2f}; "
      f"index {index:6}; "
      f"{text}"
    )


best_score("attentat sarajevo")

The 5 results come from *L'Humanité*, which provides 87% of the paragraphs and alone covers the end of June. The second one, from June 29, announces the death of the archduke heir "d adtriche" (of Austria, *Autriche*): a search on `autriche` wouldn't find it by that word.

## BM25

First try `best_score("guerre allemands")`: what do you notice about the length of the top paragraphs? The corpus is in French, so the queries stay in French.

The course presents Okapi BM25, an improvement on the TF-IDF dot product, which compares each document's length to the average length:

$$
\text{score}(D, Q)
= \sum_{i = 1}^n
\text{IDF}(q_i)
\cdot
\frac{f(q_i, D) \cdot (k_1 + 1)}
     {f(q_i, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\bar{|D|}}\right)}
$$

where:

- $q_1, \dots, q_n$ are the terms of the query $Q$
- $\text{IDF}(q_i) = \log\frac{N}{df_i}$, the one from TF-IDF: $df_i$ of the base's $N$ documents contain $q_i$
- $f(q_i, D)$ is the frequency of the term $q_i$ in the document $D$
- $|D|$ is the number of terms in $D$, and $\bar{|D|}$ the average length of the base's documents
- $k_1 = 1.5$ and $b = 0.75$

Here, a document is a paragraph. Count the terms of each paragraph with a `CountVectorizer`, in a variable `counter` and a matrix `counts`, and keep each paragraph's length in `lengths`. Then implement the function `bm25_scores`, which returns each paragraph's score for a query.

**Useful documentation**

- [`sklearn.feature_extraction.text.CountVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)
- [`scipy.sparse.csr_matrix.getnnz`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.csr_matrix.getnnz.html)
- [`numpy.log`](https://numpy.org/doc/stable/reference/generated/numpy.log.html)

In [ ]:
# Your code here


def bm25_scores(query: str, k1: float = 1.5, b: float = 0.75) -> numpy.ndarray:
  # Your code here
  ...

### Additional hints

The calls and parameters to use:

- `counter.fit_transform(data[:, 0])`: the sparse matrix `counts`, with one row per paragraph and one column per term
- `counts.getnnz(axis=0)`: for each term, the number of paragraphs containing it ($df_i$)
- `numpy.asarray(counts.sum(axis=1)).ravel()`: for each paragraph, its number of terms ($|D|$). `CountVectorizer` ignores one-letter words, so 12 paragraphs have no term at all: count them as 1 (`numpy.maximum(..., 1)`), or $b = 1$ divides by zero
- `counter.transform([query]).indices`: the columns of the query's terms. `CountVectorizer` lowercases it, splits it and ignores the words absent from the corpus
- `counts[:, term].toarray().ravel()`: a term's column, that is $f(q_i, D)$ for all the paragraphs. numpy's operations then apply to every paragraph at once, with no loop over the documents
- to check: `N` is 48,333, `lengths.mean()` about 178 and `bm25_scores("jaurès assassiné").max()` about 13.51

A skeleton to copy into the exercise cell, replacing each `...`:

```python
counter = sklearn.feature_extraction.text.CountVectorizer()
counts = ...
N = counts.shape[0]
df = ...  # df_i for each term
lengths = numpy.maximum(..., 1)  # |D| for each paragraph


def bm25_scores(query: str, k1: float = 1.5, b: float = 0.75) -> numpy.ndarray:
  # The part of the denominator that depends only on the paragraph
  normalization = k1 * (1 - b + b * ...)
  scores = numpy.zeros(N)
  for term in counter.transform([query]).indices:
    f = ...  # f(q_i, D) for all the paragraphs
    idf = ...
    scores += ...
  return scores
```

### Solution

In [ ]:
best_score("guerre allemands")

counter = sklearn.feature_extraction.text.CountVectorizer()
counts = counter.fit_transform(data[:, 0])
N = counts.shape[0]
df = counts.getnnz(axis=0)
# CountVectorizer ignores one-letter words: 12 paragraphs have no term at all, and we
# count them as 1 so as not to divide by zero when b is 1
lengths = numpy.maximum(numpy.asarray(counts.sum(axis=1)).ravel(), 1)


def bm25_scores(query: str, k1: float = 1.5, b: float = 0.75) -> numpy.ndarray:
  normalization = k1 * (1 - b + b * lengths / lengths.mean())
  scores = numpy.zeros(N)
  for term in counter.transform([query]).indices:
    f = counts[:, term].toarray().ravel()
    idf = numpy.log(N / df[term])
    scores += idf * f * (k1 + 1) / (f + normalization)
  return scores


print(bm25_scores("jaurès assassiné").max())

## Comparison with the cosine

Implement a function `compare(query: str) -> None` that displays, for the cosine similarity and then for BM25, the 5 best results: the score, the length, the index and the start of the text. A result's whole text is in `data[index, 0]`.

Then look at:

1. on `guerre allemands` (Germans war) and `jaurès assassiné` (Jaurès assassinated), the length of the paragraphs each method ranks first, against the corpus's median length. Which one brings up the article on Jaurès's assassination?
2. the length of BM25's 5 best results on `guerre allemands`, with $b$ at 0, then 0.75, then 1: what does $b$ control?
3. on `mobilisation générale` (general mobilization), the results of both methods: how many really announce a general mobilization? Why?

Conclude: what does length normalization bring, and what does it leave unsolved?

**Useful documentation**

- [`numpy.argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)
- [`numpy.median`](https://numpy.org/doc/stable/reference/generated/numpy.median.html)

In [ ]:
def compare(query: str) -> None:
  # Your code here
  ...


# Your code here

### Additional hints

The calls and parameters to use:

- `(-scores).argsort()[:5]`: the indices of the 5 best scores, as in `best_score`
- `sklearn.metrics.pairwise.cosine_similarity(X, query_vector)[:, 0]`: each paragraph's similarity with the query
- `data[index, 0][:80]`: the start of a paragraph
- `bm25_scores("guerre allemands", b=b)`: BM25 with another value of $b$
- to check: `numpy.median(lengths)` is 102

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def show_top5(scores: numpy.ndarray) -> None:
  for index in ...:  # the indices of the 5 best scores
    start = data[index, 0][:80]
    print(f"{scores[index]:5.2f}; {lengths[index]:4} terms; {index:6}; {start}")


def compare(query: str) -> None:
  print(f"Query: {query}")
  print("Cosine")
  query_vector = vectorizer.transform([query])
  show_top5(...)  # each paragraph's similarity with the query
  print("BM25")
  show_top5(...)


print("Median length:", ...)
compare("guerre allemands")
compare("jaurès assassiné")

for b in (0, 0.75, 1):
  best = ...  # the indices of BM25's 5 best scores with this b
  print(f"b = {b}: {lengths[best]}")

compare("mobilisation générale")
```

### Solution

In [ ]:
def show_top5(scores: numpy.ndarray) -> None:
  for index in (-scores).argsort()[:5]:
    start = data[index, 0][:80]
    print(f"{scores[index]:5.2f}; {lengths[index]:4} terms; {index:6}; {start}")


def compare(query: str) -> None:
  print(f"Query: {query}")
  print("Cosine")
  query_vector = vectorizer.transform([query])
  show_top5(sklearn.metrics.pairwise.cosine_similarity(X, query_vector)[:, 0])
  print("BM25")
  show_top5(bm25_scores(query))


print("Median length:", numpy.median(lengths))
compare("guerre allemands")
compare("jaurès assassiné")

for b in (0, 0.75, 1):
  best = (-bm25_scores("guerre allemands", b=b)).argsort()[:5]
  print(f"b = {b}: {lengths[best]}")

compare("mobilisation générale")

What these results show:

- The cosine similarity divides by the norm of the paragraph's vector: in a headline, the query's terms weigh heavily. On `guerre allemands`, its 5 best results have 12 to 14 terms, against a median of 102. On `jaurès assassiné`, it ranks a wine merchant first (10 terms), and the article on Jaurès (691 terms) only comes 3rd.
- BM25 ranks that article first, and the next four are also about Jaurès's death. On `guerre allemands`, two paragraphs of 84 and 85 terms enter the top 5, but three headlines stay: short texts are less favored, not ruled out.
- $b$ controls this normalization: at 0, the top 5 have 656 to 1,042 terms, because a long text contains the terms more often; at 1, BM25 goes back to headlines of 12 to 14 terms, like the cosine.
- Normalization doesn't solve everything: on `mobilisation générale`, only one result out of 5 really announces one (the Netherlands'), in both methods. A general strike in South Africa, with a mobilization of troops, comes 1st with the cosine and 2nd with BM25: a bag of words doesn't know that the two terms must be adjacent. The account of the order being posted in Paris only comes 6th with BM25, 16th with the cosine.

Three queries read by hand measure nothing: comparing two engines takes relevance judgments.